In [1]:
from datasets import load_dataset

ds = load_dataset("KisanVaani/agriculture-qa-english-only")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.97M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22615 [00:00<?, ? examples/s]

In [2]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu umap-learn matplotlib seaborn scikit-learn rank-bm25 tabulate rouge-score bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00


In [35]:
import sys
from pathlib import Path

if Path("src/config.py").exists():
    sys.path.append("src")
elif Path("../src/config.py").exists():
    sys.path.append("../src")
else:
    raise FileNotFoundError("Could not find src/config.py")

print("Using config from:", sys.path[-1])

Using config from: src


In [42]:
%cd /content
!rm -rf agriculture-rag-chatbot
!git clone https://github.com/Prathameshkadam1998/agriculture-rag-chatbot.git
%cd agriculture-rag-chatbot

/content
Cloning into 'agriculture-rag-chatbot'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 40 (delta 11), reused 24 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 15.41 KiB | 3.08 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/agriculture-rag-chatbot


In [44]:
!git pull

Already up to date.


In [45]:
!pwd
!ls
!ls src

/content/agriculture-rag-chatbot
data  models  notebooks  README.md  reports  requirements.txt  scripts	src
config.py	    generation.py  pipeline.py
data_processing.py  __init__.py    retrieval.py


In [46]:
import sys
from pathlib import Path
import importlib

SRC_PATH = Path.cwd() / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

importlib.invalidate_caches()

print("Current folder:", Path.cwd())
print("Using src path:", SRC_PATH)
print("pipeline exists:", (SRC_PATH / "pipeline.py").exists())

Current folder: /content/agriculture-rag-chatbot
Using src path: /content/agriculture-rag-chatbot/src
pipeline exists: True


In [47]:
# Imports
import re
import math
import time
import hashlib
import random
import json
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import faiss
import torch
import nltk
import umap

from tqdm import tqdm
from collections import Counter, defaultdict
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_dataset
from tabulate import tabulate
from rank_bm25 import BM25Okapi
from torch.utils.data import DataLoader
from config import (
    CHUNK_SIZE, CHUNK_OVERLAP, MAX_SENTENCES, OVERLAP_SENTENCES,
    EMBEDDING_MODELS, CROSS_ENCODER_BASE, CROSS_ENCODER_SAVE_DIR,
    EMBEDDINGS_SAVE_PATH, GENERATOR_MODEL, MAX_CONTEXT_TOKENS,
    EMBEDDING_BATCH_SIZE, EMBEDDING_SAMPLE_SIZE, FINETUNE_BATCH_SIZE,
    FINETUNE_EPOCHS, EVAL_TEST_SIZE, GEN_EVAL_SIZE, K_VALUES,
    TEST_QUERIES, TOPIC_KEYWORDS, TOPIC_COLORS, DOMAIN_STOPWORDS,
    TOPIC_QUERIES,
)

from pipeline import AgriRAGPipeline, SemanticOnlyPipeline, HybridCEPipeline

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

random.seed(42)
np.random.seed(42)